# 8.1 팀 프로젝트 미니 예시 — 데이터 선정부터 최종 측정까지 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter08_1_team_project.ipynb)

책 본문: [8.1 팀 프로젝트: 데이터 선정부터 발표까지](https://smhanlab.com/book-ml/kor/ml1/chapter08/1.html)

본 절의 "팀 프로젝트"를 실제 수업 데이터 대신 **작은 데이터 하나로 한 흐름으로**
재현하는 노트북입니다. 8.1절이 요구하는 프로젝트의 모든 단계 — 문제 정의,
베이스라인, 3분할(Ch06.3), train으로만 fit하는 전처리, Ch02~07에서
최소 두 모델 적용, val로만 튜닝, **test를 딱 한 번** 측정, 수식적
정당화 — 을 breast_cancer 데이터(569건)로 그대로 통과시켜 봅니다.
scikit-learn만 쓰며, 모든 시드가 고정되어 있습니다. (Colab에서는 첫
셀의 `IMG` 경로를 `/tmp`로 바꾸면 됩니다.)

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)

## 1. 문제 정의: "무엇을, 어떤 라벨로 예측하는가"

프로젝트의 첫 단계는 모델이 아니라 **문제를 정하는 것**이다. 여기서는
`load_breast_cancer`를 쓴다. 행(row) = 환자 한 명, 열(column) = 그
환자의 조직샘플에서 재은 30개의 특성(feature), 라벨 = 양성(0)
/ 악성(1) — "이 환자가 악성 진단을 받았는가?"를 30개의 숫자로부터
예측하는 **이진 분류** 문제다. 8.2절의 체크리스트에서 "문제 정의가
명확한가"를 따지는 이유를 생각하며, 문제의 3요소(행이 무엇인지,
라벨이 무엇인지, 지표가 무엇인지)를 먼저 확인한다.

지표 선택: 두 클래스의 비율이 ≈37:63으로 극단적이지 않아 정확도도
쓸 수 있지만, 본 프로젝트의 표준처럼 **F1**을 주 지표로 쓰고
참고로 ROC-AUC/PR-AUC를 함께 본다(Ch02.3 — 불균형이 심해지면
지표는 PR-AUC 쪽으로 바꾼다).

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X, y = data.data, data.target
print("breast_cancer:", X.shape, " (행=환자, 열=특성 30개)")
print("benign=%d  malignant=%d (악성 %.1f%%)" % (np.sum(y == 0), np.sum(y == 1), 100 * np.mean(y == 1)))

# Ch06.3의 2단계 3분할: 먼저 40%를 (val+test)로 떼고, 그 40%를 절반씩
Xtr, Xtemp, ytr, ytemp = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
Xva, Xte, yva, yte = train_test_split(Xtemp, ytemp, test_size=0.5, random_state=0, stratify=ytemp)
print("크기: train=%d  val=%d  test=%d" % (len(Xtr), len(Xva), len(Xte)))
print("train 클래스 비율: benign=%d  malignant=%d" % (np.sum(ytr == 0), np.sum(ytr == 1)))

# 3분할이 정말 서로 안 겹치는지 unique id로 기계적 점검
Xid = np.arange(len(X)).reshape(-1, 1)
Xtr_i, Xt_i, _, _ = train_test_split(Xid, np.zeros(len(X)), test_size=0.4, random_state=0, stratify=y)
Xva_i, Xte_i, _, _ = train_test_split(Xt_i, np.zeros(len(Xt_i)), test_size=0.5, random_state=0, stratify=ytemp)
s_tr, s_va, s_te = set(map(int, Xtr_i.ravel())), set(map(int, Xva_i.ravel())), set(map(int, Xte_i.ravel()))
print("3분할 pairwise disjoint:", s_tr.isdisjoint(s_va), s_tr.isdisjoint(s_te), s_va.isdisjoint(s_te))

breast_cancer: (569, 30)  (행=환자, 열=특성 30개)
benign=212  malignant=357 (악성 62.7%)
크기: train=341  val=114  test=114
train 클래스 비율: benign=127  malignant=214
3분할 pairwise disjoint: True True True


## 2. 전처리: 스케일러는 train으로만 fit

30개 특성의 스케일이 다르므로(Ch04, Ch05의 거리·마진 기반 모델에서
스케일링이 필요한 이유) 표준화를 한다. **핵심 원칙(Ch06.3)**:
`StandardScaler`의 `fit`(평균·표준편차 계산)은 **train에만** 적용하고,
val/test에는 그 통계량으로 `transform`만 한다. 전체 데이터로 fit하면
test의 정보가 전처리에 스며드는 "정규화 통계량 누수"다(6.3절
노트북 §3에서 z-값이 어떻게 달라지는지 직접 재본 바로 그 실수).

In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(Xtr)   # <- fit은 train으로만
Xtr_s, Xva_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xva), scaler.transform(Xte)
print("train 특성 0의 평균=%.4f std=%.4f (train에서만 계산)" % (scaler.mean_[0], scaler.scale_[0]))
print("train의 (변환 후) min/max =", np.round(Xtr_s.min(0)[:3], 2), np.round(Xtr_s.max(0)[:3], 2))
print("val의 (train 통계로 변환한) min/max =", np.round(Xva_s.min(0)[:3], 2), np.round(Xva_s.max(0)[:3], 2))

train 특성 0의 평균=14.1122 std=3.6705 (train에서만 계산)
train의 (변환 후) min/max = [-1.75 -2.26 -1.73] [3.81 4.67 3.81]
val의 (train 통계로 변환한) min/max = [-1.94 -1.97 -1.9 ] [2.45 3.33 2.38]


## 3. 모델 후보: Ch02~07에서 최소 두 개

8.1절의 "최소 두 가지 모델"을 넉넉히 5개로 늘려본다 — 선형
(로지스틱회귀, Ch02), 거리(kNN, Ch04), 트리 단독(Ch07.1),
앙상블(랜덤포레스트·GBDT, Ch07). 모델은 **val F1로만** 비교한다 —
test를 "살짝 훔쳐보며" 고르면 선택 편향(§6)을 먹기 때문이다.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score

cands = {
    "LogisticRegression(C=1)  [Ch02]": LogisticRegression(max_iter=2000, random_state=0),
    "kNN(k=5)                 [Ch04]": KNeighborsClassifier(n_neighbors=5),
    "DecisionTree(depth=3)    [Ch07.1]": DecisionTreeClassifier(max_depth=3, random_state=0),
    "RandomForest(100)        [Ch07.2]": RandomForestClassifier(n_estimators=100, random_state=0),
    "GBDT(100, lr=0.1)        [Ch07.3]": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=0),
}
val_f1, val_acc = {}, {}
for name, m in cands.items():
    m.fit(Xtr_s, ytr)
    val_f1[name] = f1_score(yva, m.predict(Xva_s))
    val_acc[name] = accuracy_score(yva, m.predict(Xva_s))
    print("%-32s val acc=%.3f  val F1=%.3f" % (name, val_acc[name], val_f1[name]))
best = max(val_f1, key=val_f1.get)
print("\nval F1로 고른 모델:", best, "(val F1=%.3f)" % val_f1[best])

LogisticRegression(C=1)  [Ch02]  val acc=0.947  val F1=0.958
kNN(k=5)                 [Ch04]  val acc=0.965  val F1=0.973
DecisionTree(depth=3)    [Ch07.1] val acc=0.904  val F1=0.920


RandomForest(100)        [Ch07.2] val acc=0.930  val F1=0.943


GBDT(100, lr=0.1)        [Ch07.3] val acc=0.930  val F1=0.942

val F1로 고른 모델: kNN(k=5)                 [Ch04] (val F1=0.973)


## 4. 베이스라인: "다수 클래스만 찍는" 모델 (Ch02.3)

발표 전에 반드시 보고해야 할 숫자가 하나 있다 — **무엇도 배우지
않은 모델**(train의 다수 클래스를 전부 예측)의 성능. 이 데이터에서 다수
클래스는 malignant(62.7%)다. 모델이 이 베이스라인을 못 넘기면
"모델을 만들었다"는 것이 아니다. Ch02.3의 정확도의 함정 —
극단적으로 불균형하면 베이스라인이 99%에 근접 — 도 이 숫자로
항상 점검한다(8.2절의 사기 탐지 발표 예가 정확히 이 함정).

In [5]:
# 다수 클래스 = train에서 **많이** 나온 클래스 (train 정보만 사용)
counts = np.bincount(ytr)
maj = int(np.argmax(counts))
base_val_acc = accuracy_score(yva, np.full(len(yva), maj))
base_val_f1 = f1_score(yva, np.full(len(yva), maj))
print("베이스라인(전부 '%s'로 예측 — train에서 %d/%d):" % ("malignant" if maj else "benign", counts[maj], len(ytr)))
print("  train acc=%.3f  val acc=%.3f  val F1=%.3f" %
      (accuracy_score(ytr, np.full(len(ytr), maj)), base_val_acc, base_val_f1))
print("-> val F1로 고른 모델(%s)의 val F1=%.3f는 베이스라인 val F1=%.3f보다 %.3f 높다" %
      (best.split("[")[0].strip(), val_f1[best], base_val_f1, val_f1[best] - base_val_f1))

베이스라인(전부 'malignant'로 예측 — train에서 214/341):
  train acc=0.628  val acc=0.632  val F1=0.774
-> val F1로 고른 모델(kNN(k=5))의 val F1=0.973는 베이스라인 val F1=0.774보다 0.198 높다


## 5. test 측정 — 딱 한 번

val F1로 고른 최종 모델로 test를 **단 한 번** 측정한다(Ch06.3).
이 숫자가 이 미니-프로젝트의 "최종 성능"이고, 여기서부터는 이
모델을 더 고치거나 다른 후보로 바꿀 수 없다 — test를 기준으로
무엇을 다시 하면 §6의 선택 편향을 먹는 순간이다. (실제 프로젝트
보고서도 이 숫자 하나를 '최종 성능'으로 보고하고, val의 튜닝
기록은 별개로 제시한다.)

In [6]:
from sklearn.metrics import precision_score, recall_score

best_model = cands[best].fit(Xtr_s, ytr)   # 최종 fit은 train으로만
yhat = best_model.predict(Xte_s)
yprob = best_model.predict_proba(Xte_s)[:, 1]
print("TEST (한 번만) — %s" % best)
print("  accuracy=%.3f  F1=%.3f" % (accuracy_score(yte, yhat), f1_score(yte, yhat)))
print("  ROC-AUC=%.3f  PR-AUC=%.3f  (참고 지표, Ch02.3)" %
      (roc_auc_score(yte, yprob), average_precision_score(yte, yprob)))
print("  precision=%.3f  recall=%.3f  (test malignant %d건)" %
      (precision_score(yte, yhat), recall_score(yte, yhat), np.sum(yte == 1)))

TEST (한 번만) — kNN(k=5)                 [Ch04]
  accuracy=0.939  F1=0.950
  ROC-AUC=0.980  PR-AUC=0.979  (참고 지표, Ch02.3)
  precision=0.957  recall=0.944  (test malignant 71건)


## 6. "테스트를 살짝 훔쳐봤다면"? — 선택 편향을 숫자로

만약 test를 보고 "5개 모델 중 제일 잘 나온 것"을 골랐다면? 각
후보의 test 성능을 (진짜 성능 + 잡음 \(\mathcal{N}(0,\sigma^2)\))로
모델링하면, 후보 \(m\)개 중 최고를 고르는 행위는 \(m\)번의
잡음 뽑기 중 max를 고르는 것과 같고 max는 항상 위쪽(낙관 쪽)으로
치우친다(6.3절 노트북 §2). F1 스케일로 잡음 \(\sigma=0.02\)를
가정하면:

- \(m=5\)(5개 모델 비교) → 부풀림 ≈ **0.023** — F1 0.97의 결과에서
  "실제 0.947일 수 있다"는 뜻.
- \(m=50\)(5개 모델 × 하이퍼파라미터 10개) → 부풀림 ≈ **0.045**.

"테스트를 한 번만"이 형식적 규칙이 아니라 **정량적으로** 필요한 이유다 —
본 절 §"test를 한 번만, 그리고 왜"와 같은 계산이다.

In [7]:
rng = np.random.default_rng(42)
def e_max(m, sigma, n_sims=40000):
    return rng.normal(0, sigma, size=(n_sims, m)).max(axis=1).mean()

print("후보 수 m   E[max] (진짜 F1=0, 잡음 sigma=0.02)")
for m in [1, 2, 5, 10, 20, 50, 100]:
    print("  m=%3d   %.4f   <- test로 고르면 F1이 이렇게 부풀어 오른다" % (m, e_max(m, 0.02)))

후보 수 m   E[max] (진짜 F1=0, 잡음 sigma=0.02)
  m=  1   0.0001   <- test로 고르면 F1이 이렇게 부풀어 오른다
  m=  2   0.0112   <- test로 고르면 F1이 이렇게 부풀어 오른다
  m=  5   0.0232   <- test로 고르면 F1이 이렇게 부풀어 오른다
  m= 10   0.0307   <- test로 고르면 F1이 이렇게 부풀어 오른다
  m= 20   0.0374   <- test로 고르면 F1이 이렇게 부풀어 오른다


  m= 50   0.0450   <- test로 고르면 F1이 이렇게 부풀어 오른다
  m=100   0.0501   <- test로 고르면 F1이 이렇게 부풀어 오른다


## 7. 학습 곡선: 하이퍼파라미터는 val로 정한다 (8.1절 필수 도표)

"최종 모델의 학습 곡선을 반드시 제시한다"는 요구를 GBDT의
`n_estimators`로 보여준다. train F1은 처음부터 거의 1.0(341건을
나무 몇 그루면 외우기 쉬움) — **train 곡선은 "배운 것을" 재고 val
곡선만 "재는 것을" 한다**. 그래서 `n_estimators`의 값은 train 곡선이
아니라 **val 곡선의 극대**로 정한다 — 정확히 Ch06의 검증 데이터
원칙, 그리고 Ch07.3 GBDT에서 "train 오차는 계속 줄지만 검증
성능이 나빠지면 멈추라(early stopping)"는 말의 그림이다. (이
데이터가 너무 깨끗해서 val F1이 큰 폭으로 안 벌어지는 것 자체가
"이 정도 데이터에서 GBDT는 트리가 많아도 크게 무너지지 않는다"는
정보다 — 깨끗하지 않은 실전 데이터에서 이 두 곡선이 벌어지는
모양이 Ch07.3의 과적합 이야기다.)

In [8]:
n_grid = [10, 30, 50, 80, 120, 160, 200, 250, 300]
tr_f1, va_f1 = [], []
for n in n_grid:
    m = GradientBoostingClassifier(n_estimators=n, learning_rate=0.1, random_state=0).fit(Xtr_s, ytr)
    tr_f1.append(f1_score(ytr, m.predict(Xtr_s)))
    va_f1.append(f1_score(yva, m.predict(Xva_s)))
    print("  n=%3d  train F1=%.3f  val F1=%.3f" % (n, tr_f1[-1], va_f1[-1]))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(n_grid, tr_f1, "o-", color="#1d4ed8", lw=2, label="train F1")
ax.plot(n_grid, va_f1, "s-", color="#dc2626", lw=2, label="val F1")
ax.set_xlabel("n_estimators (GBDT의 트리 개수)")
ax.set_ylabel("F1")
ax.set_title("GBDT 학습 곡선: 하이퍼파라미터(n_estimators)는 val 곡선으로 정한다")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(IMG + "/ch08_1_learning_curve.svg")
plt.show()
print("\n(본문에 ch08_1_learning_curve.svg로 삽입됨)")

  n= 10  train F1=0.995  val F1=0.950
  n= 30  train F1=1.000  val F1=0.942


  n= 50  train F1=1.000  val F1=0.934


  n= 80  train F1=1.000  val F1=0.942


  n=120  train F1=1.000  val F1=0.950


  n=160  train F1=1.000  val F1=0.950


  n=200  train F1=1.000  val F1=0.957


  n=250  train F1=1.000  val F1=0.965


  n=300  train F1=1.000  val F1=0.972

(본문에 ch08_1_learning_curve.svg로 삽입됨)


## 8. 수식적 정당화: SHAP으로 예측 하나를 분해 (Ch07.3)

8.1절의 "수식적 정당화 최소 1개" 중 하나인 SHAP 분해를, **라이브러리
없이 선택 순서만 두 개인 장난감 함수**로 정확히 손계산한다.
특징이 2개(\(A, B\))인 모델의 예측함수 \(f\)가

\[f(\varnothing)=0.50,\quad f(\{A\})=0.58,\quad f(\{B\})=0.70,\quad
f(\{A,B\})=0.78\]

일 때, 7.3절의 Shapley값 공식을 그대로 대입한다 — 특징이 2개면
순서가 \(A\!\to\!B\)와 \(B\!\to\!A\) 두 가지뿐이라 근사가 아닌
**정확한 값**을 낼 수 있다.

In [9]:
f = {frozenset(): 0.50, frozenset("A"): 0.58, frozenset("B"): 0.70, frozenset("AB"): 0.78}

# phi_j = (모든 추가 순서에서 j의 한계 기여의 평균)
phi_A = 0.5 * ((f[frozenset("A")] - f[frozenset()])      # 순서 A->B: A의 한계 기여
               + (f[frozenset("AB")] - f[frozenset("B")]))  # 순서 B->A: A의 한계 기여
phi_B = 0.5 * ((f[frozenset("B")] - f[frozenset()])      # 순서 B->A: B의 한계 기여
               + (f[frozenset("AB")] - f[frozenset("A")]))  # 순서 A->B: B의 한계 기여
print("순서 A->B: A의 한계 기여 = %.2f, B의 한계 기여 = %.2f" %
      (f[frozenset("A")] - f[frozenset()], f[frozenset("AB")] - f[frozenset("A")]))
print("순서 B->A: B의 한계 기여 = %.2f, A의 한계 기여 = %.2f" %
      (f[frozenset("B")] - f[frozenset()], f[frozenset("AB")] - f[frozenset("B")]))
print("phi_A = %.2f,  phi_B = %.2f" % (phi_A, phi_B))

# 가산성(additivity) 검증: phi_0 + phi_A + phi_B == f({A,B})
ok = abs(f[frozenset()] + phi_A + phi_B - f[frozenset("AB")]) < 1e-12
print("가산성 검산: %.2f + %.2f + %.2f = %.2f == f({A,B})=%.2f  일치=%s" %
      (f[frozenset()], phi_A, phi_B, f[frozenset()] + phi_A + phi_B, f[frozenset("AB")], ok))
assert ok

순서 A->B: A의 한계 기여 = 0.08, B의 한계 기여 = 0.20
순서 B->A: B의 한계 기여 = 0.20, A의 한계 기여 = 0.08
phi_A = 0.08,  phi_B = 0.20
가산성 검산: 0.50 + 0.08 + 0.20 = 0.78 == f({A,B})=0.78  일치=True


읽어볼 점: \(B\)가 \(A\)보다 기여가 큰(\(0.20 > 0.08\)) 이유는 **순서에
따라 한계 기여가 달라지기** 때문이다 — \(A\)가 먼저 가면 \(B\)는
\(0.70-0.50=0.20\)를, \(B\)가 먼저 가면 \(A\)는 \(0.58-0.50=0.08\)만
기여한다. Shapley값은 이 순서 의존을 **모든 순서의 평균**으로 공평하게
분배해, "기여의 합이 정확히 모델의 총 변화를 맞춘다"는 보장을
주는 것이다. 실제 프로젝트에서는 `shap` 라이브러리가 이 평균을
샘플링으로 근사해 GBDT의 **예측 하나**를 이렇게 분해한다 —
본문 §"좋은 예"가 SHAP을 쓴다고 하면, 이 작은 예가 그
분해의 뼈대다.

## 9. 이 흐름이 프로젝트의 모든 요구사항을 어떻게 만족하는가

| 8.1절 요구사항 | 이 노트북의 대응 |
|---|---|
| 정형 데이터 1개 | §1 breast_cancer(569×30) |
| 최소 두 모델 | §3에서 5개(Ch02/04/07.1/07.2/07.3) |
| train/val/test 3분할, test 한 번만 | §2, §5(+ §6: "두 번 쓰면"의 정량적 대가) |
| 전처리 통계량 train fit | §2 `fit(Xtr)` → `transform` |
| 하이퍼파라미터는 val/교차검증으로 | §7 학습 곡선(val F1) |
| 베이스라인 보고 | §4 다수 클래스 모델 |
| 수식적 정당화 ≥ 1개 | §8 Shapley값 가산성(손계산 검증) |
| 잘 안 된 부분의 정직한 보고 | §7 "train은 1.0이지만 val로만 판단"의 읽기 |

이 표가 곧 8.2절에서 다른 팀의 보고서를 볼 때 사용할 **검토
눈**이다 — 항목이 하나라도 빠진(특히 test를 두 번 쓴, 베이스라인
이 없는) 발표는 체크리스트의 "방법론 적절성"에서 바로 지적해야
할 대상이다.